# 🧠 Student Depression Detection — Exploratory Analysis

This notebook covers:
1. Dataset overview & class distribution
2. Feature distribution visualisations
3. Correlation heatmap
4. Mutual information analysis
5. SHAP summary plot
6. Model comparison


In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from preprocess import preprocess_pipeline

plt.rcParams['figure.dpi'] = 130
sns.set_theme(style='whitegrid', palette='muted')


In [ ]:
# ── Load & preprocess ─────────────────────────────────────────
DATA_PATH = '../data/student_depression.csv'
arts = preprocess_pipeline(DATA_PATH)

df           = arts['df']
X            = arts['X']
y            = arts['y']
feature_cols = arts['feature_cols']
mi_df        = arts['mi_df']
corr_df      = arts['corr_df']

feat_df = pd.DataFrame(X, columns=feature_cols)
feat_df['depression'] = y
print(feat_df.shape)
feat_df.head()


In [ ]:
# ── Class distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = pd.Series(y).value_counts()
axes[0].pie(counts, labels=['Not Depressed', 'Depressed'],
            autopct='%1.1f%%', colors=['#4CAF50', '#EF5350'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Class Distribution', fontweight='bold')

axes[1].bar(['Not Depressed', 'Depressed'], counts.values,
            color=['#4CAF50', '#EF5350'], edgecolor='white')
axes[1].set_title('Class Counts', fontweight='bold')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# ── Correlation heatmap ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 9))
corr_matrix = feat_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Mutual Information ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
top_mi = mi_df.head(13)
feats  = top_mi['feature'].str.replace('_', ' ').str.title()
ax.barh(feats[::-1], top_mi['mi_score'][::-1],
        color='#5C6BC0', edgecolor='white', alpha=0.9)
ax.set_title('Mutual Information with Depression (Target)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('MI Score')
plt.tight_layout()
plt.show()


In [ ]:
# ── Feature distributions by class ───────────────────────────
num_cols = ['cgpa', 'academic_pressure', 'financial_stress',
            'work_study_hours', 'sleep_duration']
num_cols = [c for c in num_cols if c in feature_cols]

fig, axes = plt.subplots(1, len(num_cols), figsize=(4 * len(num_cols), 4))
for ax, col in zip(axes, num_cols):
    feat_df[feat_df['depression'] == 0][col].hist(
        ax=ax, alpha=0.6, color='#4CAF50', bins=20, label='Not Depressed')
    feat_df[feat_df['depression'] == 1][col].hist(
        ax=ax, alpha=0.6, color='#EF5350', bins=20, label='Depressed')
    ax.set_title(col.replace('_', ' ').title(), fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)

fig.suptitle('Feature Distributions by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── SHAP Summary (requires trained model) ────────────────────
try:
    import pickle, shap
    with open('../models/depression_model.pkl', 'rb') as f:
        bundle = pickle.load(f)
    model = bundle['model']
    explainer = shap.TreeExplainer(model)
    # Sample 500 rows for speed
    idx = np.random.choice(len(X), min(500, len(X)), replace=False)
    shap_vals = explainer.shap_values(X[idx])
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
    shap.summary_plot(shap_vals, X[idx], feature_names=feature_cols,
                      plot_type='bar', show=True)
except Exception as e:
    print(f'SHAP summary skipped: {e}')
    print('Train the model first with: python src/train.py')
